# DeepCF 模型训练

本 notebook 用于交互式训练 DeepCF 模型。

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings("ignore")

from deepcf.config import DeepCFConfig
from deepcf.data.generator import generate_synthetic_data
from deepcf.data.utils import split_edges, scale_features
from deepcf.data.dataset import BPRDataset, collate_bpr_batch
from deepcf.model.vgae import DeepCFVGAE
from deepcf.train.trainer import Trainer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. 配置参数

调整以下参数来控制训练过程。

In [ ]:
config = DeepCFConfig()
config.data.num_nodes = 300
config.train.epochs = 100
config.train.lr = 0.001
config.train.batch_size = 128
config.model.latent_dim = 32
config.output_dir = "outputs/notebook_run"
config.seed = SEED

print(f"Nodes: {config.data.num_nodes}")
print(f"Epochs: {config.train.epochs}")
print(f"Latent dim: {config.model.latent_dim}")
print(f"LR: {config.train.lr}")

## 2. 生成合成数据

In [ ]:
data = generate_synthetic_data(
    num_nodes=config.data.num_nodes,
    num_features=config.model.input_dim,
    edge_density=config.data.edge_density,
    community_k=config.data.community_k,
    seed=config.seed,
)
X = scale_features(data["features"])
A = data["adjacency"]
W = data["weights"]
labels = data["labels"]

print(f"Edges: {int(A.sum() // 2)}")
print(f"Communities: {len(np.unique(labels))}")

## 3. 划分边与构建图张量

In [ ]:
splits = split_edges(A, train_ratio=0.85, val_ratio=0.05, seed=config.seed)
train_adj = splits["train_adj"]

edge_list = []
for i in range(config.data.num_nodes):
    for j in range(i + 1, config.data.num_nodes):
        if train_adj[i, j] > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])

edge_index_np = np.array(edge_list).T
x_tensor = torch.tensor(X, dtype=torch.float32)
edge_index_tensor = torch.tensor(edge_index_np, dtype=torch.long)
adj_true = torch.tensor(train_adj, dtype=torch.float32)

dataset = BPRDataset(train_adj, W, num_negatives=1, seed=config.seed)
dataset.x = x_tensor
dataset.edge_index = edge_index_tensor
train_loader = DataLoader(
    dataset, batch_size=config.train.batch_size,
    shuffle=True, collate_fn=collate_bpr_batch,
)

print(f"Training edges: {len(edge_list) // 2}")

## 4. 构建模型并开始训练

In [ ]:
model = DeepCFVGAE(config.model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

trainer = Trainer(model, config, train_loader)
history = trainer.train(adj_true)

print(f"
Best loss: {trainer.best_val_loss:.4f} at epoch {trainer.best_epoch}")

## 5. 训练曲线

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["epoch"], history["train_loss"], color="blue", linewidth=1.5)
ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)
ax2.plot(history["epoch"], history["lr"], color="green", linewidth=1.5)
ax2.set_title("Learning Rate"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("LR")
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Training complete!")